In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = Path("../data/raw/PS_20174392719_1491204439457_log.csv")

print(DATA_PATH.exists())
print(DATA_PATH.resolve())

True
D:\fraud-detection-ai\data\raw\PS_20174392719_1491204439457_log.csv


In [3]:
df = pd.read_csv(DATA_PATH)
df.shape

(6362620, 11)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [7]:
TARGET = "isFraud"
df[TARGET].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [8]:
df[TARGET].value_counts(normalize=True)*100

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

In [15]:
df.columns

Index(['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
       'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud'],
      dtype='str')

In [18]:
df['Origin_balance_change'] = (
    df['oldbalanceOrg'] - df['newbalanceOrig']
)
df['Destination_balance_change'] = (
    df['oldbalanceDest'] - df['newbalanceDest']
)

df['amount_to_origin_balance_ratio'] = (
    df["amount"] / (df["oldbalanceOrg"] + 1)
)

In [17]:
df["amount_to_destination_balance_ratio"] = (
    df["amount"] /
    (df["oldbalanceDest"] + 1)
)

In [19]:
df["origin_balance_error"] = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
)
df["destination_balance_error"] = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
)

In [20]:
df["log_amount"] = np.log1p(df["amount"])

In [21]:
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,Origin_balance_change,Destination_balance_change,amount_to_origin_balance_ratio,amount_to_destination_balance_ratio,origin_balance_error,destination_balance_error,log_amount
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0,9839.64,0.0,0.057834,9839.640000,0.0,9839.64,9.194276
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0,1864.28,0.0,0.087731,1864.280000,0.0,1864.28,7.531166
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0,181.00,0.0,0.994505,181.000000,0.0,181.00,5.204007
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0,181.00,21182.0,0.994505,0.008545,0.0,21363.00,5.204007
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0,11668.14,0.0,0.280788,11668.140000,0.0,11668.14,9.364703


In [23]:
df.shape

(6362620, 16)

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 16 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   step                                 int64  
 1   type                                 str    
 2   amount                               float64
 3   oldbalanceOrg                        float64
 4   newbalanceOrig                       float64
 5   oldbalanceDest                       float64
 6   newbalanceDest                       float64
 7   isFraud                              int64  
 8   isFlaggedFraud                       int64  
 9   Origin_balance_change                float64
 10  Destination_balance_change           float64
 11  amount_to_origin_balance_ratio       float64
 12  amount_to_destination_balance_ratio  float64
 13  origin_balance_error                 float64
 14  destination_balance_error            float64
 15  log_amount                           float6

In [25]:
df =pd.get_dummies(
    df,
    columns=["type"],
    drop_first=False,
    dtype = int
)

In [26]:
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

In [27]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y, test_size=0.2,random_state=42,
    stratify=y
)

In [28]:
print("Train:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest:")
print(y_test.value_counts(normalize=True) * 100)

Train:
isFraud
0    99.870926
1     0.129074
Name: proportion, dtype: float64

Test:
isFraud
0    99.870887
1     0.129113
Name: proportion, dtype: float64


In [29]:
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [30]:
train_data = X_train.copy()
train_data["isFraud"] = y_train

test_data = X_test.copy()
test_data["isFraud"] = y_test

In [31]:
train_data.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

test_data.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)

In [32]:
print(df.shape)
print(X.shape)
print(X_train.shape)
print(X_test.shape)

(6362620, 20)
(6362620, 19)
(5090096, 19)
(1272524, 19)


In [33]:
print("TRAIN")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nTEST")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True) * 100)

TRAIN
isFraud
0    5083526
1       6570
Name: count, dtype: int64
isFraud
0    99.870926
1     0.129074
Name: proportion, dtype: float64

TEST
isFraud
0    1270881
1       1643
Name: count, dtype: int64
isFraud
0    99.870887
1     0.129113
Name: proportion, dtype: float64


In [34]:
train_check = pd.read_csv("../data/processed/train.csv", nrows=5)

test_check = pd.read_csv("../data/processed/test.csv", nrows=5)

print("TRAIN COLUMNS:")
print(train_check.columns.tolist())

print("\nTEST COLUMNS:")
print(test_check.columns.tolist())

TRAIN COLUMNS:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'Origin_balance_change', 'Destination_balance_change', 'amount_to_origin_balance_ratio', 'amount_to_destination_balance_ratio', 'origin_balance_error', 'destination_balance_error', 'log_amount', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'isFraud']

TEST COLUMNS:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'Origin_balance_change', 'Destination_balance_change', 'amount_to_origin_balance_ratio', 'amount_to_destination_balance_ratio', 'origin_balance_error', 'destination_balance_error', 'log_amount', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'isFraud']


In [35]:
print("Train shape:", pd.read_csv("../data/processed/train.csv").shape)
print("Test shape:", pd.read_csv("../data/processed/test.csv").shape)


Train shape: (5090096, 20)
Test shape: (1272524, 20)
